# Transformers & Attention Mechanism

The Transformer architecture revolutionized NLP and now dominates across many domains.

1. **Self-Attention** - Scaled dot-product attention
2. **Multi-Head Attention** - Attending to different representation subspaces
3. **Transformer Encoder** - Building a classifier from scratch
4. **Positional Encoding** - Injecting sequence order

**Task**: Text classification using a Transformer encoder built from scratch in PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Scaled Dot-Product Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- **Q** (Query): What am I looking for?
- **K** (Key): What do I contain?
- **V** (Value): What information do I provide?

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding."""
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [ ]:
class TransformerClassifier(nn.Module):
    """Transformer encoder for sequence classification."""
    
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2,
                 dim_feedforward=256, num_classes=2, max_len=200, dropout=0.1):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoder = PositionalEncoding(d_model, max_len, dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )
        self.d_model = d_model
    
    def forward(self, x):
        # Create padding mask
        padding_mask = (x == 0)  # True where padded
        
        # Embed and add positional encoding
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        
        # Transformer encoder
        x = self.transformer_encoder(x, src_key_padding_mask=padding_mask)
        
        # Global average pooling over sequence (ignoring padding)
        mask = ~padding_mask.unsqueeze(-1)  # (batch, seq_len, 1)
        x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        
        return self.classifier(x)

# Example usage
model = TransformerClassifier(vocab_size=5000).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Test with dummy data
dummy_input = torch.randint(0, 5000, (4, 200)).to(device)
output = model(dummy_input)
print(f"Input shape: {dummy_input.shape}, Output shape: {output.shape}")

In [ ]:
# Visualize positional encodings
pe = PositionalEncoding(d_model=64, max_len=100, dropout=0.0)
dummy = torch.zeros(1, 100, 64)
pe_values = pe(dummy).squeeze().detach().numpy()

plt.figure(figsize=(12, 4))
plt.imshow(pe_values.T, cmap="RdBu", aspect="auto")
plt.xlabel("Position")
plt.ylabel("Dimension")
plt.title("Sinusoidal Positional Encoding")
plt.colorbar()
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Self-attention captures long-range dependencies** without the sequential bottleneck of RNNs
2. **Multi-head attention** lets the model attend to different aspects simultaneously
3. **Positional encoding** is necessary because attention is permutation-invariant
4. **Transformers parallelize better** than RNNs - much faster on GPUs
5. **Pre-trained transformers** (BERT, GPT) have made training from scratch rarely necessary for NLP
6. **Key hyperparameters**: d_model, nhead (must divide d_model), num_layers, dim_feedforward